# Subtareas: Anti-inflación de empresas + eliminación de NULLs

**Fecha**: 2026-04-29  
**Origen**: Auditoría del endpoint `/api/v1/stats/empresa`  
**Objetivo**: Blindar la tabla `empresas` contra duplicados, eliminar NULLs en `empresa_id` usando un centinela explícito, normalizar correctamente los sufijos legales, y detectar frases de anonimato del scraper.

---

## Diagnóstico actual

### Problemas detectados

| # | Problema | Consecuencia |
|---|----------|-------------|
| 1 | Regex de sufijos no captura variantes con espacios (`S. A. S`) | Empresas duplicadas con nombres casi idénticos |
| 2 | `Empresa.nombre` sin `unique=True` | Duplicados por race condition entre scrapers |
| 3 | `empresa_id` acepta NULL | Ofertas sin empresa (incoherente conceptualmente) |
| 4 | Frases de anonimato se guardan como nombres reales | Infla conteo de empresas únicas |

### Flujo actual (para referencia)

```
Scraping: get_safe_text(offer, "p.dFlex a", default="-")
  → Empresa visible: "CONATEMPO S. A. S"
  → Empresa no encontrada: "-"
  → Empresa anónima: "Importante empresa del sector"
         ↓
Cleaning: sanitize_text → handle_nulls("-" → NaN)
         ↓
Normalization: parse_company_name → strip sufijos (regex débil)
  → "CONATEMPO S. A. S" → "CONATEMPO S. A. S" (SIN cambios, el regex falla)
  → NaN → NaN
         ↓
Persistencia:
  → Si NaN → emp_id = None → NULL en DB
  → Si string → busca/crea Empresa por nombre exacto
         ↓
API: LEFT JOIN → NULLs = ratio_nulos 13.04%
```

---

## Diseño propuesto (flujo final)

```
Scraping: get_safe_text(offer, "p.dFlex a", default="Empresa no especificada")
  → Empresa visible: "CONATEMPO S. A. S"
  → Empresa no encontrada: "Empresa no especificada"
  → Empresa anónima: "Importante empresa del sector"
         ↓
Cleaning: sanitize_text → handle_nulls (ya no hay "-" para empresa)
         ↓
Normalization:
  1. parse_company_name → strip sufijos (regex CORREGIDO)
     → "CONATEMPO S. A. S" → "CONATEMPO"
     → "ACTIVOS S A S" → "ACTIVOS"
  2. normalize_company_anonymous → detecta frases anónimas
     → "Importante empresa del sector" → "Empresa no especificada"
     → "Confidencial" → "Empresa no especificada"
         ↓
Persistencia:
  → Siempre hay string (nunca NaN)
  → Busca/crea Empresa por nombre (protegido por UNIQUE en DB)
  → empresa_id NUNCA es NULL
         ↓
API: ratio_anonimas mide "Empresa no especificada" / total
```

---

## Subtareas (orden de ejecución)

| # | Archivo | Cambio | Depende de |
|---|---------|--------|------------|
| 1 | `database/models.py` | `Empresa.nombre`: agregar `unique=True, nullable=False` | — |
| 2 | `database/models.py` | `Oferta.empresa_id`: agregar `nullable=False` | — |
| 3 | `scripts/reset_db.py` (o seed) | Insertar `Empresa(id=1, nombre="Empresa no especificada")` | 1, 2 |
| 4 | `analytics/data/patterns.py` | Corregir `COMPANY_SUFFIXES_PATTERN` con `\s*` | — |
| 5 | `analytics/processes/parsing.py` | Nueva función `normalize_company_anonymous()` | 3 |
| 6 | `analytics/processes/normalization.py` | Enganchar `normalize_company_anonymous` en el pipeline | 5 |
| 7 | `analytics/processes/persistence.py` | Eliminar lógica de `emp_id = None` | 2, 3 |
| 8 | `scrapers/.../extraction.py` | Cambiar `default="-"` → `default="Empresa no especificada"` en `get_safe_text` de empresa | 3 |
| 9 | `analytics/processes/cleaning.py` | Opcional: quitar `handle_nulls` para columna empresa (ya no hay `"-"`) | 8 |

---

## Subtarea 1: `Empresa.nombre` — `unique=True, nullable=False`

**Archivo**: `database/models.py:8`

**Antes**:
```python
nombre = Column(String(255))
```

**Después**:
```python
nombre = Column(String(255), unique=True, nullable=False)
```

**Qué cambia en los datos**:
- PostgreSQL rechazará cualquier INSERT con `nombre` duplicado (IntegrityError)
- PostgreSQL rechazará cualquier INSERT con `nombre = NULL`
- Si ya hay duplicados en la tabla actual, la migración fallará → hay que limpiar primero

**Verificación previa necesaria**:
```sql
SELECT nombre, COUNT(*) FROM empresas GROUP BY nombre HAVING COUNT(*) > 1;
```
Si hay resultados, toca deduplicar manualmente antes de aplicar el cambio.

---

## Subtarea 2: `Oferta.empresa_id` — `nullable=False`

**Archivo**: `database/models.py:22`

**Antes**:
```python
empresa_id = Column(Integer, ForeignKey('empresas.id'))
```

**Después**:
```python
empresa_id = Column(Integer, ForeignKey('empresas.id'), nullable=False)
```

**Qué cambia en los datos**:
- PostgreSQL rechazará cualquier INSERT con `empresa_id = NULL`
- Si hay ofertas actuales con `empresa_id = NULL`, la migración fallará → hay que asignarles el centinela primero

**Verificación previa necesaria**:
```sql
SELECT COUNT(*) FROM ofertas WHERE empresa_id IS NULL;
```
Si > 0, toca `UPDATE ofertas SET empresa_id = 1 WHERE empresa_id IS NULL` después de crear la empresa centinela.

---

## Subtarea 3: Empresa centinela `"Empresa no especificada"`

**Archivo**: Seed inicial (puede ser en `reset_db.py` o script separado)

**Valor final**:
```python
Empresa(id=1, nombre="Empresa no especificada")
```

**Qué cambia en los datos**:
- Se crea UNA sola fila en `empresas` con id=1
- Todas las ofertas que hoy tienen `empresa_id = NULL` se actualizan a `empresa_id = 1`
- `ratio_nulos_empresa` del endpoint `/empresa` pasará a reflejar ofertas ligadas a esta empresa centinela
- El valor final en JSON será `"Empresa no especificada"` en vez de NULL

**Script SQL de migración** (a ejecutar después del seed):
```sql
UPDATE ofertas SET empresa_id = 1 WHERE empresa_id IS NULL;
```

---

## Subtarea 4: Corregir regex de sufijos legales

**Archivo**: `analytics/data/patterns.py:27`

**Antes**:
```python
COMPANY_SUFFIXES_PATTERN = re.compile(
    r'\b(S\.?A\.?S\.?|L\.?T\.?D\.?A\.?|S\.?A\.?|I\.?N\.?C\.?|B\.?I\.?C\.?)\b',
    re.IGNORECASE
)
```

**Después**:
```python
COMPANY_SUFFIXES_PATTERN = re.compile(
    r'\b(S\.?\s*A\.?\s*S\.?|L\.?\s*T\.?\s*D\.?\s*A\.?|S\.?\s*A\.?|I\.?\s*N\.?\s*C\.?|B\.?\s*I\.?\s*C\.?)\b',
    re.IGNORECASE
)
```

**Qué cambia en los datos**:
- `"CONATEMPO S. A. S"` → `"CONATEMPO"` (ANTES no se normalizaba)
- `"ACTIVOS S A S"` → `"ACTIVOS"` (ANTES no se normalizaba)
- `"EMPRESA S.A.S."` → `"EMPRESA"` (ya funcionaba, sigue funcionando)
- `"EMPRESA LTDA"` → `"EMPRESA"` (ya funcionaba, sigue funcionando)
- `"EMPRESA L T D A"` → `"EMPRESA"` (ANTES no se normalizaba)

**Valores finales esperados**:
| Entrada | Salida |
|---|---|
| `CONATEMPO S. A. S` | `CONATEMPO` |
| `CONATEMPO S.A.S.` | `CONATEMPO` |
| `ACTIVOS S A S` | `ACTIVOS` |
| `Gi Group Colombia` | `Gi Group Colombia` |
| `Estrategia Segura` | `Estrategia Segura` |
| `Empresa no especificada` | `Empresa no especificada` |

---

## Subtarea 5: Nueva función `normalize_company_anonymous()`

**Archivo**: `analytics/processes/parsing.py` (nueva función)

**Lista de frases anónimas a detectar**:
```python
ANONYMOUS_COMPANY_PHRASES = [
    "Importante empresa del sector",
    "Importante empresa",
    "Empresa del sector",
    "Confidencial",
    "Empresa confidencial",
    "Por definir",
    "A convenir",
    "No especificada",
    "No especificado",
]
```

**Código propuesto**:
```python
ANONYMOUS_COMPANY_PHRASES = [
    "importante empresa del sector",
    "importante empresa",
    "empresa del sector",
    "confidencial",
    "empresa confidencial",
    "por definir",
    "a convenir",
    "no especificada",
    "no especificado",
]

def normalize_company_anonymous(text):
    """Detecta frases de anonimato y las reemplaza por el centinela oficial."""
    if pd.isna(text):
        return "Empresa no especificada"
    clean = str(text).strip().lower()
    for phrase in ANONYMOUS_COMPANY_PHRASES:
        if phrase in clean:
            return "Empresa no especificada"
    return text
```

**Qué cambia en los datos**:
- `"Importante empresa del sector"` → `"Empresa no especificada"`
- `"Confidencial"` → `"Empresa no especificada"`
- `"Por definir"` → `"Empresa no especificada"`
- `"CONATEMPO"` → `"CONATEMPO"` (sin cambios)

---

## Subtarea 6: Enganchar `normalize_company_anonymous` en el pipeline

**Archivo**: `analytics/processes/normalization.py:19-23` y `analytics/pipeline.py:19`

**Cambio en `normalization.py`**:
```python
def normalize_companies(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia sufijos legales y unifica empresas anónimas."""
    if 'empresa' in df.columns:
        df['empresa'] = df['empresa'].apply(parse_company_name)
        df['empresa'] = df['empresa'].apply(normalize_company_anonymous)
    return df
```

**Cambio en `pipeline.py`**: Agregar el import de `normalize_company_anonymous`.

**Orden de aplicación**:
1. Primero strip sufijos (`parse_company_name`)
2. Luego detectar anónimas (`normalize_company_anonymous`)

---

## Subtarea 7: Ajustar `persistence.py` — eliminar `emp_id = None`

**Archivo**: `analytics/processes/persistence.py:47-55`

**Antes**:
```python
emp_name = row.get("empresa")
emp_id = None
if emp_name and not pd.isna(emp_name):
    emp = session.query(Empresa).filter_by(nombre=str(emp_name).strip()).first()
    if not emp:
        emp = Empresa(nombre=str(emp_name).strip())
        session.add(emp)
        session.commit()
    emp_id = emp.id
```

**Después**:
```python
emp_name = row.get("empresa")
if not emp_name or pd.isna(emp_name):
    emp_name = "Empresa no especificada"
emp_name = str(emp_name).strip()
emp = session.query(Empresa).filter_by(nombre=emp_name).first()
if not emp:
    emp = Empresa(nombre=emp_name)
    session.add(emp)
    session.commit()
emp_id = emp.id
```

**Qué cambia**: Ya no existe el caso `emp_id = None`. Si la empresa no se encuentra, se usa el centinela.

---

## Subtarea 8: Cambiar default del scraper para empresa

**Archivo**: `scrapers/computrabajo/processes/extraction.py:54`

**Antes**:
```python
"empresa": await get_safe_text(offer, "p.dFlex a"),
# get_safe_text usa default="-"
```

**Después**:
```python
"empresa": await get_safe_text(offer, "p.dFlex a", default="Empresa no especificada"),
```

**Qué cambia**: Cuando el scraper no encuentra el nombre de empresa, en vez de `"-"`, se usa directamente `"Empresa no especificada"`. Esto elimina la necesidad de que `handle_nulls` convierta `"-"` → `NaN` para esta columna.

---

## Subtarea 9 (opcional): Quitar `handle_nulls` para columna empresa

**Archivo**: `analytics/processes/cleaning.py:24-28`

Ya no es necesario convertir `"-"` a `NaN` para la columna `empresa` porque el scraper ahora usa `"Empresa no especificada"` como default. `handle_nulls` sigue siendo útil para otras columnas (`ubicacion`, `salario`, etc.) que sí pueden tener `"-"`.

**Sin cambios en cleaning.py** — se deja como está para el resto de columnas.

---

## Resumen: valores finales en el output de `/empresa`

| Campo | Antes | Después |
|---|---|---|
| `total_empresas_unicas` | 51 (incluye frases anónimas como empresas reales) | ~48 (sin las anónimas) |
| `top_10_empresas` | Incluye `"CONATEMPO S. A. S"` y `"ACTIVOS S A S"` | Muestra `"CONATEMPO"` y `"ACTIVOS"` normalizados |
| `ratio_nulos_empresa` | `"13.04%"` (NULLs) | `"X%"` (ofertas con `"Empresa no especificada"`) |
| `empresas_solo_ingles` | Puede incluir empresas anónimas | Excluye `"Empresa no especificada"` |
| `analisis_larga_cola` | Inflado por empresas anónimas y duplicados | Más preciso |

---

## Orden de ejecución recomendado

```
Paso 1: Subtarea 4 (regex)          ← sin dependencias, se puede testear solo
Paso 2: Subtarea 5 (función anónimas) ← depende de 4 conceptualmente
Paso 3: Subtarea 6 (enganchar en pipeline) ← depende de 5
Paso 4: Subtarea 8 (scraper default) ← sin dependencias
Paso 5: Subtarea 1 (unique nombre)   ← requiere verificar que no hay duplicados
Paso 6: Subtarea 2 (nullable FK)     ← requiere verificar que no hay NULLs
Paso 7: Subtarea 3 (seed centinela)  ← depende de 1, 2
Paso 8: Subtarea 7 (persistencia)    ← depende de 2, 3
Paso 9: Subtarea 9 (opcional)        ← depende de 8
```

**Nota**: Los pasos 1-4 (normalización) se pueden ejecutar y testear sin tocar la DB. Los pasos 5-8 requieren migración de datos existentes.